# 06 - Segmented devices: tapers and fabrication variations

Real devices are rarely uniform: film thickness and linewidth drift across the chip, and some designs
vary the line parameters on purpose (tapers). `twpasolver` describes such devices as a chain of
uniform **segments**. This notebook shows how to build a segmented device, checks the segmented solver
against the uniform one, and uses it for random fabrication variations and for a taper.

We start again from the KI-TWPA holotype `imskit_v3_holotype.json` (3WM, $I_{dc}$ bias, pump at 15.8 GHz,
just above the first stopband), with the extended 3WM mode set of notebook 04.

In [ ]:
import logging
import time

import matplotlib.pyplot as plt
import numpy as np

from twpasolver import ModeSet, TWPAnalysis
from twpasolver.cmes.solver_config import BlochSolverConfig
from twpasolver.logger import log
from twpasolver.models import TWPA

log.setLevel(logging.ERROR)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

FREQS = (0.05, 66, 2e-3)  # GHz; reaches the 4th pump harmonic (63.2 GHz)
FP = 15.8  # pump frequency (GHz)
SIG = np.linspace(3, 12.5, 20)  # signal frequencies (GHz)
MODES = ModeSet.extended_3wm(
    n_pump_harmonics=3, n_frequency_conversion=2, n_signal_harmonics=0
)
CFG = BlochSolverConfig(passes=60)

base = TWPA.from_file("imskit_v3_holotype.json")
print(
    f"{base.N} supercells of {base.N_tot // base.N:.0f} unit cells, {base.N_tot:.0f} unit cells in total"
)

## 1. Building a segmented device

A segmented device is a `TWPA` whose `cells` are themselves `TWPA` objects. Each segment is a uniform
TWPA with its own unit cells, number of supercells `N` and nonlinearity (`Istar`, `Idc`); the outer
`TWPA` (with `N=1`) only chains them and carries the pump current and the reference impedance.
Each segment must hold a **whole number of supercells**: the Bloch modes of a segment are defined on
its supercell, so a segment cannot end in the middle of one. To cut the holotype into `n_seg` equal
segments, `n_seg` must divide its 1200 supercells.

`make_segmented` below copies the holotype's unit cells into `n_seg` segments and scales the line
inductance $L$ and the finger inductance $L_f$ of every unit cell of segment $j$ by `L_scale[j]` and
`Lf_scale[j]`.

In [ ]:
def make_segmented(n_seg, L_scale=None, Lf_scale=None):
    if base.N % n_seg:
        raise ValueError(
            f"n_seg={n_seg} must divide N={base.N} (whole supercells per segment)"
        )
    L_scale = np.ones(n_seg) if L_scale is None else L_scale
    Lf_scale = np.ones(n_seg) if Lf_scale is None else Lf_scale
    segments = []
    for j in range(n_seg):
        cells = [c.model_copy(deep=True) for c in base.cells]
        for c in cells:
            c.L *= L_scale[j]
            c.Lf *= Lf_scale[j]
        segments.append(base.model_copy(update={"cells": cells, "N": base.N // n_seg}))
    return TWPA(cells=segments, N=1, Ip0=base.Ip0, Z0_ref=base.Z0_ref)


def analysis(twpa):
    return TWPAnalysis(twpa=twpa, freqs=FREQS)


an = analysis(make_segmented(4))
lin = an.linear
print("k shape (n_segments, n_freq):", lin.k.shape)
print("segment bounds (unit cells):", lin.seg_bounds)
print("segment midpoints (unit cells):", lin.seg_midpoints)

`an.linear` stores the Bloch wavenumber `k`, attenuation `alpha`, Bloch impedance `Zb` and its
`reflection_coeff` **per segment**, with shape `(n_segments, n_freq)` (a uniform device is one
segment). `S21`, `S11` and `abcd` are those of the whole chain, computed by multiplying the
segment ABCD matrices, so they already contain the reflections at the junctions. `stopbands` lists the
stopbands of the *first* segment only.

The two gain models treat the junctions differently:

* `model="bloch"` solves each segment on its own forward and backward Bloch modes and joins consecutive
  segments with **exact junction conditions**, from continuity of voltage and current at the shared
  supercell boundary. A step in the line parameters therefore reflects part of every wave, and these
  reflections interfere with each other and with the ports.
* `model="simple"` has forward waves only. It interpolates the per-segment wavenumbers (and the
  nonlinearity) **linearly between segment midpoints**, so it sees a smooth taper without reflections.
  It describes how phase matching drifts along the line, not the reflections.

## 2. Check: identical segments and runtime

Cutting the holotype into identical segments changes nothing physically, so the `bloch` gain must
equal that of the uniform device: the junctions between identical segments are transparent. We
compare a few `n_seg` and time the gain sweep (20 signals, after a warm-up call that compiles the
Numba kernels).

In [ ]:
def bloch_gain(an, **kwargs):
    return an.gain(SIG, pump=FP, model="bloch", modes=MODES, bloch_config=CFG, **kwargs)


an_uni = analysis(base)
bloch_gain(an_uni)  # warm-up (compilation)
t0 = time.perf_counter()
r_uni = bloch_gain(an_uni)
t_uni = time.perf_counter() - t0
print(
    f"uniform: max gain {r_uni.gain_db.max():.2f} dB, {t_uni:.2f} s, "
    f"not converged at {SIG[~r_uni.converged]} GHz"
)

n_list = [1, 5, 20, 60, 200]
timing, dev = [], []
for n in n_list:
    a = analysis(make_segmented(n))
    a.linear  # linear response outside the timing
    times = []
    for _ in range(2):  # best of two runs
        t0 = time.perf_counter()
        r = bloch_gain(a)
        times.append(time.perf_counter() - t0)
    timing.append(min(times))
    dev.append(np.max(np.abs(r.gain_db - r_uni.gain_db)))
    print(
        f"n_seg={n:4d} ({base.N // n:4d} supercells each): {timing[-1]:5.2f} s, "
        f"max |gain - uniform| = {dev[-1]:.3f} dB, {np.sum(~r.converged)} not converged"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(n_list, timing, "o-")
ax.set_ylim(bottom=0)
ax.set(
    xlabel="number of segments",
    ylabel="time per gain sweep (s)",
    title=f"{len(SIG)} signals, bloch model",
)
fig.tight_layout()

The identical-segment results agree with the uniform device to within a few hundredths of a dB. The
small differences grow with the number of segments: they stay below the tolerance of the
forward-backward iteration (`convergence_threshold`, a relative residual of $10^{-2}$) and are largest
near the edges of the gain band, where the iteration converges more slowly. Points that do not
converge within the 60 passes are printed above; treat them with care.

The cost grows slowly with the number of segments: the integrator is restarted and the mixing terms
are rebuilt in every segment, but segments with identical unit cells share the same Bloch tables,
which are computed only once. Wall-clock timings on a shared machine are noisy; take the plot as an
order of magnitude.

## 3. Fabrication variations

Now every segment gets its own random $L$ and $L_f$: each is multiplied by $1+\sigma\,\mathcal N(0,1)$,
independently per segment and per parameter, with $\sigma = 2\%$ and 40 segments of 30 supercells.
$L$ sets the impedance and the phase velocity of the line, so its steps reflect; $L_f$ is in the
fingers' shunt branch and mostly changes the dispersion at high frequency, around the stopbands.
Three seeds give three "chips".

In [ ]:
n_seg, sigma = 40, 0.02
chips = {}
for seed in [1, 2, 3]:
    rng = np.random.default_rng(seed)
    L_scale = 1 + sigma * rng.standard_normal(n_seg)
    Lf_scale = 1 + sigma * rng.standard_normal(n_seg)
    a = analysis(make_segmented(n_seg, L_scale, Lf_scale))
    r = bloch_gain(a)
    chips[seed] = {"L": L_scale, "Lf": Lf_scale, "an": a, "r": r}
    f_ps = FP + SIG[-1]  # p+s at the highest signal
    in_gap = np.sum(a.linear.alpha[:, np.argmin(np.abs(a.linear.freqs - f_ps))] > 1e-4)
    print(
        f"seed {seed}: max gain {r.gain_db.max():.2f} dB (nominal {r_uni.gain_db.max():.2f} dB), "
        f"{np.sum(~r.converged)} points not converged; p+s = {f_ps:.1f} GHz in the stopband of "
        f"{in_gap}/{n_seg} segments, S21 there {r.gain_db[-1]:.1f} dB"
    )

In [ ]:
f = an_uni.linear.freqs
win = (f > 12.8) & (f < 15.0)
colors = ["C0", "C1", "C2"]
fig, axs = plt.subplots(2, 2, figsize=(13, 8))
edges = np.linspace(0, 1, n_seg + 1)
for c, (seed, d) in zip(colors, chips.items()):
    axs[0, 0].stairs(
        100 * (d["L"] - 1), edges, baseline=None, color=c, label=f"seed {seed}"
    )
    axs[0, 0].stairs(100 * (d["Lf"] - 1), edges, baseline=None, color=c, ls=":")
axs[0, 0].set(
    xlabel="position $x/\\ell$",
    ylabel="change (%)",
    title="$L$ (solid) and $L_f$ (dotted)",
)
axs[0, 0].legend(fontsize=9)

# Attenuation per unit cell of every segment of chip 1: nonzero inside each segment's stopband.
lin1 = chips[1]["an"].linear
for j, alpha in enumerate(lin1.alpha):
    axs[0, 1].plot(f[win], alpha[win], color=plt.cm.viridis(j / n_seg), lw=0.8)
axs[0, 1].plot(f[win], an_uni.linear.alpha[0][win], "k--", lw=1.5, label="nominal")
axs[0, 1].axvline(FP, color="r", ls=":", label="pump")
axs[0, 1].set(
    xlabel="frequency (GHz)",
    ylabel="alpha (1 / unit cell)",
    title="first stopband of each segment, seed 1",
)
axs[0, 1].legend(fontsize=9)

axs[1, 0].plot(SIG, r_uni.gain_db, "k", lw=2, label="nominal")
for c, (seed, d) in zip(colors, chips.items()):
    r = d["r"]
    axs[1, 0].plot(SIG, r.gain_db, color=c, label=f"seed {seed}")
    axs[1, 0].plot(SIG[~r.converged], r.gain_db[~r.converged], "x", color=c)
axs[1, 0].set(
    xlabel="signal frequency (GHz)",
    ylabel="$S_{21}$ (dB)",
    title="gain (x: not converged)",
)
axs[1, 0].legend(fontsize=9)

axs[1, 1].plot(SIG, r_uni.gain_db_11, "k", lw=2, label="nominal")
for c, (seed, d) in zip(colors, chips.items()):
    axs[1, 1].plot(SIG, d["r"].gain_db_11, color=c, label=f"seed {seed}")
axs[1, 1].set(
    xlabel="signal frequency (GHz)",
    ylabel="$S_{11}$ (dB)",
    title="signal reflection at the input",
)
axs[1, 1].legend(fontsize=9)
fig.tight_layout()

A few-percent spread moves the stopband of each segment by a few hundred MHz (top right): the
stopband frequency scales roughly as $1/\sqrt{LC}$, and the pump at 15.8 GHz sits 1.5 GHz above its upper
edge, where the dispersion is strong. The pump wavenumber, and with it the phase matching, then changes
from segment to segment. For this spread the gain in the band changes only by a few tenths of a dB from
chip to chip (bottom left). The input
reflection $S_{11}$ under pump (bottom right) is dominated by the amplified signal reflected at the
device ends, which the nominal device has too; the steps between segments change it from chip to
chip.

Modes far from the signal band are sensitive too: at the highest signal frequency the sum-frequency
mode $p+s$ lies just above the second stopband of the nominal device, and the spread puts it inside the
stopband of some segments (counted in the output above). The signal transmission at that point then differs strongly
from chip to chip. Note that the plotted stopbands come from `an.linear.alpha` per segment, while
`an.linear.stopbands` only reports those of the first segment.

## 4. A taper

Finally, a deliberate linear ramp of $L_f$ from its nominal value at the input to 80% at the output,
over 20 segments. To separate the effect of the steps from that of the ramp itself, we compare:

* the same ramp with 40 segments (a finer staircase of the same taper);
* an abrupt step with the same end values ($L_f$ at 100% in the first half, 80% in the second);
* the `simple` model on the 20-segment ramp, which interpolates the taper smoothly and has no reflections.

In [ ]:
taper = {
    "ramp, 20 segments": make_segmented(20, Lf_scale=np.linspace(1, 0.8, 20)),
    "ramp, 40 segments": make_segmented(40, Lf_scale=np.linspace(1, 0.8, 40)),
    "step, 2 segments": make_segmented(2, Lf_scale=np.array([1, 0.8])),
}
res = {}
for name, tw in taper.items():
    a = analysis(tw)
    res[name] = (a, bloch_gain(a))
    print(
        f"{name}: max gain {res[name][1].gain_db.max():.2f} dB, "
        f"{np.sum(~res[name][1].converged)} points not converged"
    )
a_ramp = res["ramp, 20 segments"][0]
r_simple = a_ramp.gain(SIG, pump=FP, model="simple", modes=MODES)
r_simple_uni = an_uni.gain(SIG, pump=FP, model="simple", modes=MODES)
print(
    f"simple model: ramp {r_simple.gain_db.max():.2f} dB, uniform {r_simple_uni.gain_db.max():.2f} dB"
)

In [ ]:
lin_r, lin_0 = a_ramp.linear, an_uni.linear
band = (f > 3) & (f < 12.5)
print(
    f"Bloch impedance at 8 GHz, first/last segment: {lin_r.Zb[0, np.argmin(np.abs(f - 8))].real:.2f} / "
    f"{lin_r.Zb[-1, np.argmin(np.abs(f - 8))].real:.2f} Ohm"
)
print(
    f"max change of the linear |S21| in 3-12.5 GHz: {np.max(np.abs(lin_r.S21_db - lin_0.S21_db)[band]):.3f} dB"
)


def k_at(lin, fx):
    # Bloch wavenumber of every segment at the frequency fx (rad / unit cell)
    return np.array([np.interp(fx, f, k) for k in lin.k])


fs = FP / 2  # degenerate signal
dk_3wm = k_at(lin_r, FP) - k_at(lin_r, fs) - k_at(lin_r, FP - fs)
dk_shg = 2 * k_at(lin_r, FP) - k_at(lin_r, 2 * FP)
dk_3wm_0 = (k_at(lin_0, FP) - k_at(lin_0, fs) - k_at(lin_0, FP - fs))[0]
dk_shg_0 = (2 * k_at(lin_0, FP) - k_at(lin_0, 2 * FP))[0]
L_seg = lin_r.seg_N_tot[0]
print(
    f"change of the 3WM mismatch over the line, times the segment length: "
    f"{np.sum(dk_3wm - dk_3wm_0) * L_seg:.2f} rad"
)

fig, axs = plt.subplots(2, 2, figsize=(13, 8))
far = (f > 1) & (f < 50)
axs[0, 0].plot(f[far], 1e3 * (lin_r.k[-1] - lin_0.k[0])[far], "C3")
for sb in lin_0.stopbands[:4]:
    axs[0, 0].axvspan(*sb, color="0.85")
axs[0, 0].set(
    ylim=(-2, 2),
    xlabel="frequency (GHz)",
    ylabel="$\\Delta k$ ($10^{-3}$ rad / unit cell)",
    title="k of the last segment minus nominal k (grey: stopbands)",
)

edges20 = np.linspace(0, 1, 21)
axs[0, 1].stairs(
    1e3 * (dk_3wm - dk_3wm_0),
    edges20,
    baseline=None,
    color="C0",
    label="$k_p - k_s - k_i$, $f_s = f_p/2$",
)
axs[0, 1].stairs(
    1e3 * (dk_shg - dk_shg_0),
    edges20,
    baseline=None,
    color="C1",
    label="$2k_p - k_{2p}$",
)
axs[0, 1].set(
    xlabel="position $x/\\ell$",
    ylabel="change ($10^{-3}$ rad / unit cell)",
    title="change of the linear phase mismatch, ramp",
)
axs[0, 1].legend(fontsize=9)

axs[1, 0].plot(SIG, r_uni.gain_db, "k", lw=2, label="nominal")
for c, (name, (a, r)) in zip(["C3", "C1", "C0"], res.items()):
    axs[1, 0].plot(SIG, r.gain_db, color=c, label=name)
axs[1, 0].plot(SIG, r_simple.gain_db, "C3--", lw=1, label="ramp, simple model")
axs[1, 0].set(xlabel="signal frequency (GHz)", ylabel="$S_{21}$ (dB)", title="gain")
axs[1, 0].legend(fontsize=9)

axs[1, 1].plot(SIG, r_uni.gain_db_11, "k", lw=2, label="nominal")
for c, (name, (a, r)) in zip(["C3", "C1", "C0"], res.items()):
    axs[1, 1].plot(SIG, r.gain_db_11, color=c, label=name)
axs[1, 1].set(
    xlabel="signal frequency (GHz)",
    ylabel="$S_{11}$ (dB)",
    title="signal reflection at the input",
)
axs[1, 1].legend(fontsize=9)
fig.tight_layout()

$L_f$ sits in series with the finger capacitance, so it sets the finger (shunt) resonance near 100 GHz and
above, far from the signal band. Lowering it leaves the low-frequency Bloch impedance and the linear
$|S_{21}|$ in the signal band nearly unchanged (by less than 0.3 dB, printed above): the taper hardly reflects. What it
changes is the dispersion, by an amount that grows with frequency (top left): small at the signal and the
pump, larger at the pump harmonics and the sum-frequency modes. The changes of $k$ are only
$10^{-4}$ rad per unit cell, but they add up over 40800 unit cells, so the phase matching of the
amplification and that of the pump second-harmonic generation drift along the line (top right), and the
gain drops and narrows (bottom left). This notebook does not separate the two contributions.

The 20- and 40-segment staircases give practically the same gain, and the `simple` model on the same
device follows the `bloch` result within about 0.6 dB in the gain band, close to the offset between the two
models for the nominal device: here the junction reflections are a minor effect and the gain is set by the
phase-matching chirp. The abrupt step with the same end values gives a similar gain. The input
$S_{11}$ under pump falls together with the gain, since it is mainly the amplified signal reflected at
the output port. A ramp of $L$ instead changes both the impedance and the phase velocity, and moves the
stopband next to the pump: it is a much stronger perturbation than $L_f$ for this design.

When modelling a graded device with a staircase, check convergence by doubling `n_seg`, as done here: a
staircase of equal segments is itself periodic and can produce weak spurious Bragg reflections that
shrink as the segments get shorter.